# 01 - Bronze: Azure SQL master and lookup tables
Reads `tower_master` (161 rows), `network_topology_master` (170) and `region_lookup` (179) **from Azure SQL**.

Before this notebook: run `source_setup/azure_sql_create_seed_tables.sql` in Azure SQL and load the three seed CSVs into the tables.
The handbook says the CSVs are seed inputs, so Bronze must read the Azure SQL tables, not the CSV files.

Run **one** of the two read options (A or B), then run the load cell.

The Bronze copy is a snapshot: each run replaces it. Every row gets `source_system`, `ingest_ts` and a `row_hash` of all source columns.

In [ ]:
from pyspark.sql import functions as F

cat = "capstone6_telecom"

# source table in Azure SQL -> Bronze table, and the row count in the seed file
masters = {
    "tower_master":            ("bronze_tower_master",            161),
    "network_topology_master": ("bronze_network_topology_master", 170),
    "region_lookup":           ("bronze_region_lookup",           179),
}

## Option A: Lakehouse Federation (foreign catalog)

In [ ]:
foreign = "azuresql_cat"      # name of your foreign catalog

def read_master(src):
    return spark.table(f"{foreign}.dbo.{src}")

## Option B: plain JDBC
Fill in your details. Do not commit the real password to GitHub.

In [ ]:
server   = "<yourserver>.database.windows.net"
database = "<yourdatabase>"
user     = "<sql-user>"
password = "<sql-password>"

jdbc_url = (f"jdbc:sqlserver://{server}:1433;database={database};encrypt=true;"
            "trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30")

def read_master(src):
    return (spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", f"dbo.{src}")
            .option("user", user)
            .option("password", password)
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
            .load())

## Load into Bronze

In [ ]:
for src, (target, expected) in masters.items():
    df = read_master(src)
    cols = df.columns
    out = (df
           .withColumn("source_system", F.lit(f"azure_sql.dbo.{src}"))
           .withColumn("ingest_ts", F.current_timestamp())
           .withColumn("row_hash", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in cols]), 256)))
    out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{cat}.bronze.{target}")
    n = spark.table(f"{cat}.bronze.{target}").count()
    print(f"{target}: {n} rows (source file has {expected})")

In [ ]:
display(spark.table(f"{cat}.bronze.bronze_tower_master").limit(5))